# Clinical Covariates & Survival Analysis

This notebook is structured in two connected stages:

**Stage A — Covariate Screening (Cell 3)**
Fits a penalised multivariate Cox Proportional-Hazards model to a panel of clinical covariates and flags those independently and significantly associated with overall survival (OS). The significant covariates are exported as `sig_covariate_names` so Stage B can consume them automatically.

**Stage B — IPTW-Adjusted Kaplan–Meier Survival Analysis (Cell 4)**
For each combination of target gene × molecular subtype, splits patients into high/low expression groups (by IQR), fits a logistic propensity score model using the covariates flagged in Stage A, computes Inverse Probability of Treatment Weighting (IPTW), runs a weighted multivariable Cox model, and renders a Kaplan–Meier panel plot.

> **To adapt this notebook to a new dataset, edit only Cell 2 (User Configuration). No other cell needs touching.**


In [ ]:
# Under Construction


## 1. Imports


In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
import statsmodels.formula.api as smf

# Suppress the expected non-integer propensity-score warning from lifelines.
warnings.filterwarnings("ignore", message="It looks like your weights are not integers")


## 2. ⚙️ User Configuration — Edit This Cell When Changing Datasets

**Every** dataset-specific name in this notebook lives here. When you switch to a new cohort, update the values below and re-run; no other cell needs to be touched.

| Variable | What it controls |
|---|---|
| `PATH_METADATA` | Path to the clinical/metadata CSV |
| `PATH_EXPRESSION` | Path to the gene-activity/expression matrix CSV |
| `COL_BARCODE` | Column in metadata that holds sample barcodes for merging |
| `COL_VITAL_STATUS` | Column with survival status text (e.g. `'dead'` / `'alive'`) |
| `STATUS_DEAD_VALUE` | The string in `COL_VITAL_STATUS` that means deceased |
| `COL_DAYS_TO_DEATH` | Column with days-to-death (for deceased patients) |
| `COL_DAYS_TO_FOLLOWUP` | Column with days-to-last-follow-up (for censored patients) |
| `CATEGORICAL_COVARIATES` | Categorical clinical columns to screen in Stage A |
| `NUMERICAL_COVARIATES` | Numerical clinical columns to screen in Stage A |
| `ALWAYS_IN_PROPENSITY` | Covariates forced into the propensity model regardless of p-value |
| `COL_NODE_STAGING` | Raw nodal-staging column used to engineer the binary indicator |
| `NODE_VALUE_OF_INTEREST` | The specific node-stage string to binarise (e.g. `'N1b'`) |
| `TARGET_GENES` | Gene names (as they appear in the expression matrix) to analyse |
| `COL_SUBTYPE` | Column holding the molecular subtype label |
| `SUBTYPES_TO_ANALYZE` | Which subtype values to produce KM panels for |


In [ ]:
# ============================================================
#  FILE PATHS
# ============================================================
PATH_METADATA    = "./metadata.csv"
PATH_EXPRESSION  = "./activity_matrix.csv" # or count_matrix.csv

# ============================================================
#  SAMPLE / BARCODE KEY
# ============================================================
# Column in the metadata table that uniquely identifies each sample;
# used to merge metadata with the expression matrix.
COL_BARCODE = '.'

# ============================================================
#  SURVIVAL OUTCOME COLUMNS
# ============================================================
# Column containing the vital status text label.
COL_VITAL_STATUS    = '.'

# The exact string in COL_VITAL_STATUS that indicates death (case-insensitive).
STATUS_DEAD_VALUE   = '.'

# Days-to-death column (used when the patient is deceased).
COL_DAYS_TO_DEATH   = '.'

# Days-to-last-follow-up column (used when the patient is censored / alive).
COL_DAYS_TO_FOLLOWUP = '.'

# ============================================================
#  CLINICAL COVARIATES TO SCREEN (STAGE A)
# ============================================================
# Categorical columns: will be one-hot encoded before Cox regression.
CATEGORICAL_COVARIATES = ['.']

# Numerical columns: used as-is in Cox regression.
NUMERICAL_COVARIATES = ['.']

# ============================================================
#  PROPENSITY MODEL FORCED CONFOUNDERS
# ============================================================
# These covariate names are always included in the Stage B propensity
# formula regardless of whether they reached p < 0.05 in Stage A.
# List numerical column names or pre-engineered binary indicator names.
ALWAYS_IN_PROPENSITY = [
    '.',  # universal clinical confounder in most TCGA cohorts
]

# ============================================================
#  SURVIVAL ANALYSIS TARGETS (STAGE B)
# ============================================================
# Gene identifiers exactly as they appear in the expression matrix rows.
TARGET_GENES = ['.']

# Column holding the molecular subtype label used to stratify KM panels.
COL_SUBTYPE = '.'

# Which subtype values to produce KM panels for (one column per value).
SUBTYPES_TO_ANALYZE = ['.', '.']

# ============================================================
#  DERIVED NAMES  (computed automatically — do not edit)
# ============================================================
# Name of the binary indicator column engineered from COL_NODE_STAGING.
COL_NODE_INDICATOR = f"{COL_NODE_STAGING}_{NODE_VALUE_OF_INTEREST}"

print("Configuration loaded:")
print(f"  Metadata  : {PATH_METADATA}")
print(f"  Expression: {PATH_EXPRESSION}")
print(f"  Survival  : {COL_VITAL_STATUS} / {COL_DAYS_TO_DEATH} / {COL_DAYS_TO_FOLLOWUP}")
print(f"  Node indicator column: {COL_NODE_INDICATOR}")
print(f"  Target genes: {TARGET_GENES}")
print(f"  Subtypes    : {SUBTYPES_TO_ANALYZE}")


## 3. Stage A — Multivariate Cox Covariate Screening

Identifies which clinical covariates independently and significantly predict overall survival, so only those variables are carried forward as confounders in the propensity model in Stage B.

**Pipeline:**
1. Load `PATH_METADATA` and derive `OS_Event` / `OS_Time` from the survival columns configured in Cell 2.
2. Select `CATEGORICAL_COVARIATES` + `NUMERICAL_COVARIATES` and drop rows with any missing value.
3. One-hot encode categorical columns (`drop_first=True` avoids the dummy variable trap) and cast everything to `float`.
4. Drop near-constant dummy columns (variance < 1%) to prevent complete separation, which would cause the Cox partial-likelihood to diverge.
5. Fit a penalised Cox model (`penalizer=0.1` acts as an L2 ridge).
6. Extract significant covariates (p < 0.05) and store their encoded column names in `sig_covariate_names` for Stage B.


In [ ]:
# ==========================================
# 1. LOAD DATASET
# ==========================================
df = pd.read_csv(PATH_METADATA)

# ==========================================
# 2. ENGINEER OVERALL SURVIVAL OUTCOMES
# ==========================================
# Binary event indicator: STATUS_DEAD_VALUE -> 1, anything else -> 0.
df['OS_Event'] = df[COL_VITAL_STATUS].apply(
    lambda x: 1 if str(x).strip().lower() == STATUS_DEAD_VALUE.lower() else 0
)

# Survival time: days_to_death for deceased patients, days_to_last_follow_up
# for censored (alive) patients.
df['OS_Time'] = np.where(
    df['OS_Event'] == 1,
    df[COL_DAYS_TO_DEATH],
    df[COL_DAYS_TO_FOLLOWUP]
)

# Drop rows with missing or non-positive time (Cox models require T > 0).
df = df.dropna(subset=['OS_Time', 'OS_Event'])
df = df[df['OS_Time'] > 0]

# ==========================================
# 3. SELECT INDEPENDENT COVARIATES
# ==========================================
columns_to_keep = (
    ['OS_Time', 'OS_Event']
    + CATEGORICAL_COVARIATES
    + NUMERICAL_COVARIATES
)
analysis_df = df[columns_to_keep].dropna()

# ==========================================
# 4. ONE-HOT ENCODING & SINGULARITY PREVENTION
# ==========================================
# drop_first=True drops one level per variable to avoid perfect collinearity
# (the 'dummy variable trap').
encoded_df = pd.get_dummies(analysis_df, columns=CATEGORICAL_COVARIATES, drop_first=True)
encoded_df = encoded_df.astype(float)

# Near-constant dummies (variance < 1%) can cause 'complete separation',
# where the Cox partial likelihood cannot be maximised — drop them.
variance_threshold = 0.01
low_variance_cols = [
    col for col in encoded_df.columns
    if encoded_df[col].var() < variance_threshold
    and col not in ['OS_Time', 'OS_Event']
]
if low_variance_cols:
    print(f"Dropping {len(low_variance_cols)} low-variance columns:")
    print(low_variance_cols, "\n")
    encoded_df = encoded_df.drop(columns=low_variance_cols)

# ==========================================
# 5. FIT PENALISED COX MODEL
# ==========================================
# penalizer=0.1: L2 ridge penalty that shrinks unstable estimates under
# multicollinearity without fully excluding any covariate.
cph = CoxPHFitter(penalizer=0.1)
cph.fit(encoded_df, duration_col='OS_Time', event_col='OS_Event')
cph.print_summary()

# ==========================================
# 6. EXTRACT & EXPORT SIGNIFICANT COVARIATES -> STAGE B
# ==========================================
results = cph.summary[['coef', 'exp(coef)', 'p']]
results.columns = ['Coefficient', 'Hazard_Ratio', 'p_value']
significant_results = results[results['p_value'] < 0.05]

print("\n" + "="*50)
print("COVARIATES SIGNIFICANTLY CORRELATED WITH SURVIVAL (p < 0.05):")
print("="*50)
if not significant_results.empty:
    print(significant_results)
else:
    print("No covariates met p < 0.05 with the current penalizer setting.")

# Store the encoded column names of significant covariates as a module-level
# variable; Stage B (Cell 4) reads this to build its propensity formula.
sig_covariate_names = significant_results.index.tolist()
print("\nSignificant covariate names passed to Stage B:", sig_covariate_names)


## Stage A → Stage B Wiring

`sig_covariate_names` (produced above) drives the propensity formula built at the top of Cell 4. The mapping logic:

- **Numerical covariates** (listed in `NUMERICAL_COVARIATES`): the encoded column name equals the raw column name, so they are added to the formula directly.
- **The node-staging binary indicator** (`COL_NODE_INDICATOR`, e.g. `ajcc_pathologic_n_N1b`): Cell 4 pre-engineers this column *before* the propensity formula is assembled, so if any dummy derived from `COL_NODE_STAGING` is significant it can be used directly.
- **`ALWAYS_IN_PROPENSITY`** terms are included unconditionally.


## 4. Stage B — IPTW-Adjusted Kaplan–Meier Survival Analysis

For each `TARGET_GENE` × `SUBTYPES_TO_ANALYZE` combination:
1. Load and merge metadata + expression, derive survival time/event.
2. Pre-engineer the `COL_NODE_INDICATOR` binary column.
3. Build the propensity formula from `sig_covariate_names` + `ALWAYS_IN_PROPENSITY`.
4. Split into high (> Q3) / low (< Q1) expression groups.
5. Fit a logistic propensity score model and compute IPTW weights.
6. Fit a weighted multivariable Cox model → adjusted HR and p-value.
7. Draw IPTW-weighted KM curves; save the panel figure.


In [ ]:
# ==========================================
# 1. LOAD FILES
# ==========================================
# sep=None + engine='python' auto-detects delimiter (comma or tab).
df_meta    = pd.read_csv(PATH_METADATA,   sep=None, engine='python')
df_expr_raw = pd.read_csv(PATH_EXPRESSION, sep=None, engine='python')

# Strip accidental whitespace from column headers.
df_meta.columns    = df_meta.columns.str.strip()
df_expr_raw.columns = df_expr_raw.columns.str.strip()

# ==========================================
# 2. TRANSPOSE THE EXPRESSION MATRIX
# ==========================================
# The activity matrix has genes as rows and samples as columns;
# transpose so rows = samples and columns = genes, matching metadata.
gene_col_name = df_expr_raw.columns[0]  # first column holds gene identifiers
df_expr_raw[gene_col_name] = df_expr_raw[gene_col_name].astype(str).str.strip()

df_expr = df_expr_raw.set_index(gene_col_name).T
df_expr.columns.name = None
df_expr = df_expr.reset_index().rename(columns={'index': 'Barcode_Key'})

# ==========================================
# 3. DERIVE SURVIVAL TIME & EVENT
# ==========================================
def calculate_survival_time(row):
    """Return days_to_death for deceased patients, days_to_last_follow_up otherwise."""
    status = str(row[COL_VITAL_STATUS]).strip().lower()
    if status == STATUS_DEAD_VALUE.lower():
        return pd.to_numeric(row[COL_DAYS_TO_DEATH],    errors='coerce')
    else:
        return pd.to_numeric(row[COL_DAYS_TO_FOLLOWUP], errors='coerce')

df_meta['Time']  = df_meta.apply(calculate_survival_time, axis=1)
df_meta['Event'] = (
    df_meta[COL_VITAL_STATUS].astype(str).str.strip().str.lower()
    == STATUS_DEAD_VALUE.lower()
).astype(int)

# ==========================================
# 4. MERGE METADATA AND EXPRESSION
# ==========================================
# Join on stripped barcode strings to avoid whitespace-induced mismatches.
df_meta['Merge_Key'] = df_meta[COL_BARCODE].astype(str).str.strip()
df_expr['Merge_Key'] = df_expr['Barcode_Key'].astype(str).str.strip()

df_master_raw = pd.merge(df_meta, df_expr, on='Merge_Key', how='inner')

# ==========================================
# 5. GENERAL DATA CLEANING
# ==========================================
# Strip whitespace from the raw node-staging column.
df_master_raw[COL_NODE_STAGING] = df_master_raw[COL_NODE_STAGING].astype(str).str.strip()

# Pre-engineer the binary node indicator so it exists before the propensity
# formula is assembled. The column name is stored in COL_NODE_INDICATOR.
df_master_raw[COL_NODE_INDICATOR] = np.where(
    df_master_raw[COL_NODE_STAGING] == NODE_VALUE_OF_INTEREST, 1, 0
)

df_master_raw['Time']             = pd.to_numeric(df_master_raw['Time'])
df_master_raw['Event']            = pd.to_numeric(df_master_raw['Event'])
df_master_raw['age_at_diagnosis'] = pd.to_numeric(df_master_raw['age_at_diagnosis'])
df_master_raw[COL_SUBTYPE]        = df_master_raw[COL_SUBTYPE].astype(str).str.strip()

# ==========================================
# 6. BUILD PROPENSITY FORMULA FROM STAGE A
# ==========================================
# Terms that can be used directly in a statsmodels formula string:
# numerical covariates (identical encoded name) and the pre-engineered
# binary node indicator.
direct_formula_terms = set(NUMERICAL_COVARIATES) | {COL_NODE_INDICATOR}

# Start with the forced confounders from ALWAYS_IN_PROPENSITY.
propensity_terms = list(ALWAYS_IN_PROPENSITY)

# Add any significant covariate from Stage A that maps to a usable term.
for col in sig_covariate_names:
    if col in direct_formula_terms and col not in propensity_terms:
        propensity_terms.append(col)

propensity_formula = 'Is_High_Expression ~ ' + ' + '.join(propensity_terms)
print(f"Propensity formula (from Stage A): {propensity_formula}")

# Cox adjustment terms mirror the propensity terms (excluding the outcome).
cox_adjustment_terms = [t for t in propensity_terms if t != 'Is_High_Expression']

# ==========================================
# 7. INITIALISE MULTIPANEL CANVAS
# ==========================================
# Grid: one row per target gene, one column per subtype.
n_rows = len(TARGET_GENES)
n_cols = len(SUBTYPES_TO_ANALYZE)
fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(13, 7 * n_rows), sharey=True)

# Normalise axes to always be a 2-D array (handles 1-gene or 1-subtype edge cases).
if n_rows == 1:
    axes = axes[np.newaxis, :]
if n_cols == 1:
    axes = axes[:, np.newaxis]

kmf = KaplanMeierFitter()

# ==========================================
# 8. MULTI-GENE & SUBTYPE PIPELINE LOOP
# ==========================================
for row_idx, target_gene in enumerate(TARGET_GENES):

    # Coerce expression values to numeric and drop rows missing any required column.
    required_cols = ['Time', 'Event', 'age_at_diagnosis', target_gene, COL_SUBTYPE, COL_NODE_STAGING]
    df_master = df_master_raw.dropna(subset=required_cols).copy()
    df_master[target_gene] = pd.to_numeric(df_master[target_gene], errors='coerce')
    df_master = df_master.dropna(subset=[target_gene])
    df_master = df_master[df_master['Time'] >= 0]

    print(f"Gene {target_gene} — baseline cohort size: {len(df_master)} samples")

    for col_idx, subtype in enumerate(SUBTYPES_TO_ANALYZE):
        ax = axes[row_idx, col_idx]

        # Isolate the current subtype group.
        df_sub = df_master[df_master[COL_SUBTYPE] == subtype].copy()

        if len(df_sub) == 0:
            print(f"  Warning: subtype '{subtype}' not found for {target_gene}. Skipping panel.")
            ax.axis('off')
            continue

        # --- IQR-based expression split ------------------------------------
        # Patients below Q1 -> low-expression group.
        # Patients above Q3 -> high-expression group.
        # The middle two quartiles are excluded for a cleaner contrast.
        q1 = df_sub[target_gene].quantile(0.25)
        q3 = df_sub[target_gene].quantile(0.75)

        low_grp  = df_sub[df_sub[target_gene] < q1].copy()
        high_grp = df_sub[df_sub[target_gene] > q3].copy()

        # Binary treatment indicator required by the logit propensity model.
        low_grp['Is_High_Expression']  = 0
        high_grp['Is_High_Expression'] = 1

        df_cox_input = pd.concat([low_grp, high_grp]).reset_index(drop=True)

        # Guard against groups too small for the logit model to converge.
        if len(low_grp) < 3 or len(high_grp) < 3:
            print(f"  Warning: too few samples for {target_gene} ({subtype}). Skipping.")
            ax.axis('off')
            continue

        # --- Propensity score estimation -----------------------------------
        # logit model: P(Is_High_Expression=1 | covariates).
        # Formula was built dynamically in Step 6 from Stage A results.
        propensity_model = smf.logit(propensity_formula, data=df_cox_input).fit(disp=0)
        df_cox_input['Propensity_Score'] = propensity_model.predict(
            df_cox_input[propensity_terms]
        )
        # Clip away from 0/1 to avoid division-by-zero in IPTW.
        df_cox_input['Propensity_Score'] = df_cox_input['Propensity_Score'].clip(0.01, 0.99)

        # --- IPTW weight computation ----------------------------------------
        # High: weight = 1 / P(treated)
        # Low:  weight = 1 / (1 - P(treated))
        # This balances covariate distributions between groups as if
        # assignment to high/low expression had been randomised.
        df_cox_input['IPTW_Weight'] = np.where(
            df_cox_input['Is_High_Expression'] == 1,
            1.0 / df_cox_input['Propensity_Score'],
            1.0 / (1.0 - df_cox_input['Propensity_Score'])
        )

        high_grp_weighted = df_cox_input[df_cox_input['Is_High_Expression'] == 1]
        low_grp_weighted  = df_cox_input[df_cox_input['Is_High_Expression'] == 0]

        # --- Multivariable Cox model ----------------------------------------
        # Adjusts for Is_High_Expression and the Stage-A confounders together,
        # so the hazard ratio for expression is independent of those variables.
        cph = CoxPHFitter(penalizer=0.05)
        cox_features = ['Time', 'Event', 'Is_High_Expression'] + cox_adjustment_terms
        cph.fit(df_cox_input[cox_features], duration_col='Time', event_col='Event', robust=True)

        hazard_ratio     = cph.hazard_ratios_['Is_High_Expression']
        adjusted_p_value = cph.summary.loc['Is_High_Expression', 'p']

        # Format p-value: 4-decimal for >= 0.001, scientific notation otherwise.
        if adjusted_p_value >= 0.001:
            pval_str = f"{adjusted_p_value:.4f}"
        else:
            pval_str = f"{adjusted_p_value:.1e}".replace('e-0', 'e-').replace('e', '×10')
        hr_str = f"{hazard_ratio:.2f}"

        # --- IPTW-weighted Kaplan-Meier curves -----------------------------
        # The IPTW_Weight column scales each patient's contribution to the
        # KM step function, effectively creating a pseudo-population with
        # balanced covariates.
        kmf.fit(
            durations=high_grp_weighted['Time'],
            event_observed=high_grp_weighted['Event'],
            weights=high_grp_weighted['IPTW_Weight']
        )
        kmf.plot_survival_function(ci_show=True, color='#1f77b4', linewidth=2, ax=ax, label='High')

        kmf.fit(
            durations=low_grp_weighted['Time'],
            event_observed=low_grp_weighted['Event'],
            weights=low_grp_weighted['IPTW_Weight']
        )
        kmf.plot_survival_function(ci_show=True, color='#ff7f0e', linewidth=2, ax=ax, label='Low')

        # --- Panel styling -------------------------------------------------
        # Strip pipeline suffixes (_NC, _SIG) from the gene name for the title.
        gene_clean = target_gene.replace('_NC', '').replace('_SIG', '')
        ax.set_title(
            f"{gene_clean} ({subtype})\n(HR={hr_str}, p={pval_str})",
            fontsize=20, pad=10
        )
        ax.set_xlabel('Time from Sampling (Days)', fontsize=14)
        ax.set_ylabel('Overall Survival Probability', fontsize=14)
        ax.tick_params(axis='both', labelsize=12)
        ax.legend(fontsize=12, loc='lower left' if col_idx == 0 else 'upper right')
        ax.set_ylim(0, 1.05)
        ax.grid(False)

plt.tight_layout()
plt.savefig(out_path, dpi=300, bbox_inches='tight')
print(f"Saved figure to: {out_path}")
